<a href="https://colab.research.google.com/github/krupnitskyirostyslav/AB-Testing-Statistical-Analysis/blob/main/A_B_Testing_Statistical_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Connecting to Google Cloud and creating SQL request

In [ ]:
!pip install --upgrade google-cloud-bigquery

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd

In [ ]:
auth.authenticate_user()

In [ ]:
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
query = """
with session_info as (
  select
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
  from `DA.ab_test` ab
  join `DA.session` s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` sp
  on sp.ga_session_id = ab.ga_session_id
),

session_with_orders as (
  select
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group,
          count(distinct o.ga_session_id) as session_with_orders
  from `DA.order` o
  join session_info
  on o.ga_session_id = session_info.ga_session_id
  group by
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group
),

events as (
  select
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group,
          ep.event_name,
          count(ep.ga_session_id) as events_cnt
  from `DA.event_params` ep
  join session_info
  on ep.ga_session_id = session_info.ga_session_id
  group by
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group,
          ep.event_name
),

session as (
  select
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group,
          count(distinct session_info.ga_session_id) as session_cnt
  from session_info
  group by
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group
),

account as (
  select
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group,
          count(distinct acs.ga_session_id) as account_cnt
  from `DA.account_session` acs
  join session_info
  on acs.ga_session_id = session_info.ga_session_id
  group by
          session_info.date,
          session_info.country,
          session_info.device,
          session_info.continent,
          session_info.channel,
          session_info.test,
          session_info.test_group
)

SELECT
      session_with_orders.date,
      session_with_orders.country,
      session_with_orders.device,
      session_with_orders.continent,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session with orders' as event_name,
      session_with_orders.session_with_orders as value
FROM session_with_orders

UNION ALL
SELECT
      events.date,
      events.country,
      events.device,
      events.continent,
      events.channel,
      events.test,
      events.test_group,
      events.event_name,
      events.events_cnt as value
FROM events

UNION ALL
SELECT
      session.date,
      session.country,
      session.device,
      session.continent,
      session.channel,
      session.test,
      session.test_group,
      'session' as event_name,
      session.session_cnt as value
FROM session

UNION ALL
SELECT
      account.date,
      account.country,
      account.device,
      account.continent,
      account.channel,
      account.test,
      account.test_group,
      'new_account' as event_name,
      account.account_cnt as value
FROM account
"""

In [ ]:
query_job = client.query(query) # performance of the query
results = query_job.result() # waiting the query to finish
df = results.to_dataframe()

In [ ]:
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-02,Tunisia,desktop,Africa,Organic Search,2,1,new_account,1
1,2020-11-04,Jordan,desktop,Asia,Social Search,2,1,new_account,1
2,2020-11-05,New Zealand,desktop,Oceania,Undefined,2,1,new_account,1
3,2020-11-06,Serbia,desktop,Europe,Organic Search,2,1,new_account,1
4,2020-11-06,New Zealand,desktop,Oceania,Paid Search,2,1,new_account,1


#Calculating convertions and statistical significance

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest


pivot_df = df.pivot_table(
    index=["test", "test_group", "country", "device", "continent", "channel", "date"],
    columns="event_name",
    values="value",
    aggfunc="sum",
    fill_value=0
).reset_index()

conversion_events = [
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "new_account"
]

DENOMINATOR = "session"

conversion_df = pivot_df.copy()

for event in conversion_events:
    conversion_df[f"{event}_per_session"] = np.where(
        conversion_df[DENOMINATOR] > 0,
        conversion_df[event] / conversion_df[DENOMINATOR],
        0
    )

def ab_test_significance(data, events, group_col="test_group", test_col="test"):
    results = []

    for test in data[test_col].unique():
        df_test = data[data[test_col] == test]
        groups = df_test[group_col].unique()

        if len(groups) != 2:
            continue

        g1, g2 = groups
        df_a = df_test[df_test[group_col] == g1]
        df_b = df_test[df_test[group_col] == g2]

        for event in events:
            # A
            num_a = df_a[event].sum()
            den_a = df_a[DENOMINATOR].sum()
            conv_a = num_a / den_a if den_a > 0 else 0

            # B
            num_b = df_b[event].sum()
            den_b = df_b[DENOMINATOR].sum()
            conv_b = num_b / den_b if den_b > 0 else 0

            # % зміна
            lift = ((conv_b - conv_a) / conv_a * 100) if conv_b > 0 else 0

            # Z-test
            stat, pval = proportions_ztest(
                count=np.array([num_a, num_b]),
                nobs=np.array([den_a, den_b])
            )

            results.append({
                "test_number": test,
                "metric": f"{event}/session",
                "events(A)": num_a,
                "sessions(A)": den_a,
                "conversion(A)": conv_a,
                "events(B)": num_b,
                "sessions(B)": den_b,
                "conversion(B)": conv_b,
                "lift%": lift,
                "z_stat": stat,
                "p_value": pval,
                "significant": pval < 0.05
            })

    return pd.DataFrame(results)


final_df = ab_test_significance(pivot_df, conversion_events)
final_df.head(16)

,test_number,metric,events(A),sessions(A),conversion(A),events(B),sessions(B),conversion(B),lift%,z_stat,p_value,significant
0,1,add_payment_info/session,1988,45362,0.043825,2229,45193,0.049322,12.542021,-3.924884,0.000087,True
1,1,add_shipping_info/session,3034,45362,0.066884,3221,45193,0.071272,6.560481,-2.603571,0.009226,True
2,1,begin_checkout/session,3784,45362,0.083418,4021,45193,0.088974,6.660587,-2.978783,0.002894,True
3,1,new_account/session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,1.542883,0.122859,False
4,2,add_payment_info/session,2344,50637,0.046290,2409,50244,0.047946,3.576911,-1.240994,0.214608,False
5,2,add_shipping_info/session,3480,50637,0.068724,3510,50244,0.069859,1.650995,-0.709557,0.477979,False
6,2,begin_checkout/session,4262,50637,0.084168,4313,50244,0.085841,1.988164,-0.952898,0.340642,False
7,2,new_account/session,4165,50637,0.082252,4184,50244,0.083274,1.241934,-0.588793,0.556000,False
8,3,add_payment_info/session,3623,70047,0.051722,3697,70439,0.052485,1.474630,-0.643172,0.520112,False
9,3,add_shipping_info/session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,1.413727,0.157442,False


In [ ]:
# Exporting data to create a Dashboard in Tableau

from google.colab import drive

drive.mount("/content/drive")

final_df.to_csv("/content/drive/MyDrive/Mate_academy/Tasks/Portfolio/Project2/significance_data.csv", index=False)

#Summary

Statistical significance indicates whether the difference between two groups in an experiment is real and not the result of random variation. It is typically assessed using a z-statistic and a p-value, where a p-value below 0.05 indicates that the observed effect is unlikely to have arisen by chance. In practice, analysts compare the number of events and sessions between test groups, calculate conversion rates, and then apply a statistical test of proportions to determine whether the increase is significant. Google Colab is often used for this workflow because it allows you to run reproducible Python code to aggregate data, calculate conversion rates, and test significance in a clean, shareable environment.

#[Tableau Dashboards](https://public.tableau.com/views/ABtestwithstatisticalsignificance/Dashboard1?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

#[A/B test data CSV](https://drive.google.com/file/d/1PA7hj7ahPUgFt6uBxYsHTpdKIiQLSIte/view?usp=drive_link)

#[Significance data CSV](https://drive.google.com/file/d/1ZbS6W2wiWFq6nwBQ2oMjZ_-ddMtdIV7J/view?usp=drive_link)